In [ ]:
from nocode_robot_programming.state_decision.utils import kill_other_ipykernels
kill_other_ipykernels(force=True)
import trajectory_data
import matplotlib.pyplot as plt
from nocode_robot_programming.state_decision.dataloader import TrajectoryDataset
from nocode_robot_programming.state_decision.dino_model import DINOFeaturePresence
from nocode_robot_programming.state_decision.dino_with_mil import DINOWithMIL
from nocode_robot_programming.state_decision.dino_model_by_chatgpt import DINOProtoPresence
# from nocode_robot_programming.state_decision.dino_model_v2 import DINOFeaturePresence
# from nocode_robot_programming.state_decision.dino_model_v3 import DINOFeaturePresence
from nocode_robot_programming.state_decision.SIFT_model import StateDeciderSIFT
from nocode_robot_programming.state_decision.AEGP_model import AEGP
from nocode_robot_programming.state_decision.state_decider import StateDeciderBase
from nocode_robot_programming.state_decision.utils import Filename
from gesture_detector.utils import pretty_confusion_matrix
import torch
import numpy as np

from trajectory_data.skill_visualizer import play_video
from nocode_robot_programming.state_decision.utils import add_tag
from nocode_robot_programming.state_decision.utils import visualize_accuracies
from nocode_robot_programming.state_decision.dataloader import ImageDatasetView, saved_img_processing
from nocode_robot_programming.jupyter_plot import jupyter_plot as ipt, show_gray_video_cuda

seed = 48
np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

from nocode_robot_programming.state_decision.dataset_d1 import d1_peg_pick, d2_peg_pick, d1_probe, d2_probe, dupl, get_dataset_view
datafileloader = TrajectoryDataset(trajectory_data.package_path)
datasets = d1_peg_pick(datafileloader)

# Some images from the dataset

In [ ]:
d_train, d_test, d_text = datasets[0]
# d_train.showcase(fps=20)
# d_train.showcase_captions(fps=20)
d_train.play_video(fps=2, scale=1.0)
# d_test.play_video(fps=10)

# Checkpoint 2025-10-16

- Train and test accuracy achieved 100%

In [ ]:
train_stats, test_stats, model_names = [], [], []

for decider in [
        # DINOFeaturePresence(dino_variant= "dinov2_vits14", percentile_keep=None, input_size=224), # best model
        # DINOWithMIL(dino_variant= "dinov2_vits14", percentile_keep=None, input_size=224, att_hidden = 128, 
        #     dropout_p = 0.1, lr = 1e-4, weight_decay = 1e-3, epochs = 2000),
        # DINOFeaturePresence(percentile_keep=0.0, dino_variant="dinov2_vitg14", input_size=224),
        # DINOProtoPresence(percentile_keep=0.0),
        # DINOStateDecider(dino_variant = "dinov2_vitl14", percent_keep=0),
        StateDeciderSIFT(),
        # AEGP(),
    ]:
    train_stats_model, test_stats_model = [], []
    dataset_names = []
    for task_dataset_gen in [d1_peg_pick, d2_peg_pick, d1_probe, d2_probe]:
        
        for d_train, d_test, d_text in task_dataset_gen(datafileloader):
            print(f"Model: {decider.__class__.__name__}, Dataset: {d_text}")
            
            decider.train(d_train.X, d_train.y_int, d_train.y_cls); ipt.save() # loss fig obtained in train function

            y_pred = decider.predict_many(d_train.X)
            pretty_confusion_matrix.pp_matrix_from_string_data(d_train.y_names, y_pred, name=f"d_train,{decider}", show=False); ipt.save()
            train_stats_model.append((np.array(d_train.y_names) == np.array(y_pred)).mean())

            y_pred = decider.predict_many(d_test.X)
            pretty_confusion_matrix.pp_matrix_from_string_data(d_test.y_names, y_pred, name=f"d_test,{decider}", show=False); ipt.save()
            test_stats_model.append((np.array(d_test.y_names) == np.array(y_pred)).mean())
            
            # ipt.show(small=True)
            ipt.delete() # don't plot
            dataset_names.append(d_text.split(",")[0])

    train_stats.append(train_stats_model)
    test_stats.append(test_stats_model)
    model_names.append(decider.__str__())

In [ ]:
visualize_accuracies(100 * np.array(train_stats), 100 * np.array(test_stats), model_names, dataset_names, out_dir="plot")
f"Overall accuracy on train data: {round(100 * np.array(train_stats).mean())}% and test data {round(100 * np.array(test_stats).mean())}%"

- `DINOFeaturePresence(percentile_keep=0.0, input_size=112),` 78%
- `DINOFeaturePresence(percentile_keep=0.0, dino_variant="dinov2_vitg14", input_size=112),` 90%
- `DINOFeaturePresence(percentile_keep=0.0, input_size=224),` 93%
- `DINOFeaturePresence(percentile_keep=0.0, dino_variant="dinov2_vitg14", input_size=224),` 85%



#### Play all frames that are wrongly predicted

- you can modify speed with `fps`
- you can modify window size with `scale`

In [ ]:
play_video(
    d_test.X[np.array(d_test.y_names) != np.array(decider.predict_many(d_test.X))].cpu().numpy() * 256,
    fps=2, window_name="win", backend="cv2", scale=10.0)

In [ ]:
play_video(
    d_train.X[np.array(d_train.y_names) != np.array(decider.predict_many(d_train.X))].cpu().numpy() * 256,
    fps=2, window_name="win", backend="cv2", scale=10.0)

# TODO:

- Good training depends on good starting point, it doesn't have to converge

### See if accuracy varies further from the branch timestep

In [ ]:
plt.hist(d_test.Xt[np.array(d_test.y_names) == np.array(decider.predict_many(d_test.X))].cpu().numpy())